In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
from dash.exceptions import PreventUpdate

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Import re for regular expression patterns including re.compile
import re


#Import Python module and import AnimalShelter class
from AnimalShelterModule import AnimalShelter

###########################
# Data Manipulation / Model
###########################

#Username and password parameters for instantiating CRUD Python object
username = "aacuser"
password = "JREgeneric"

# Connect to database via CRUD Module
db = AnimalShelter(user= username, pwd= password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('AnimalDashboard')

#Add in Grazioso Salvare’s logo
image_filename = 'GraziosoSalvareLogo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#Place the HTML image tag in the line below into the app.layout code according to your design
#Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

#Main dashboard layout
app.layout = html.Div([
    #Display Grazioso Salvare's logo
    html.Center([
        html.A(
            href = 'https://www.snhu.edu', target = "blank",
            children=[
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height':'100px', 'width':'100px', 'textAlign':'center', 'margin-left':'70px','display':
                          'inline-block'}
                )]),
        html.B(
            style={'font-size':'10px','display':'inline-block','margin-left': '70px',
                   'verticalAlign':'top'}, 
               children=[      
        #Add project name to dashboard
            html.H1('Animal Rescue Dashboard'),
        #Add personal identifier to dashboard
            html.H2('by Jerris English, SNHU')
        ])]),
    html.Hr(),
    #Add interactive radio buttons for filtering dogs suitable for rescue training
    html.Center(html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label':html.Div(['Water Rescue'],
                                  style={'width':'300px','verticalAlign':'top'}),'value': 'Water'},
                {'label':html.Div(['Mountain/Wilderness Rescue'],
                                  style={'width':'300px','verticalAlign':'top'}), 'value': 'Mountain'},
                {'label':html.Div(['Disaster Rescue & Tracking'],
                                  style={'width':'300px','verticalAlign':'top'}),'value': 'Disaster'},
                {'label':html.Div(['Reset'],
                                  style={'width':'300px','verticalAlign':'top'}), 'value': 'All'},
            ],inline = True
        ),
    )),
    html.Hr(),
    #Add dash datatable with formatted options/settings
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        #Adding table features to allow vertical and horizontal scrolling for records
        style_table={'height': '300px', 'overflowY': 'scroll','overflowX': 'scroll'},
        #User cannot edit data
        editable=False,
        #Adds filtering for data
        filter_action="native",
        #Adds sorting for column data
        sort_action="native",
        #Allows for mulitple column sorting
        sort_mode="multi",
        #Allows one column to be selected at a time
        column_selectable=False,
        #Allows one row to be selected at a time
        row_selectable="single",
        #Disables row deletion
        row_deletable=False,
        #Sets inital selected columns to none
        selected_columns=[],
        #Sets intial selected row to first row document
        selected_rows=[0],
        #Paging logic is handled natively
        page_action="native",
        #Sets starting page at first document
        page_current=0,
        #Limits each page to 10 documents
        page_size=10,
    ),
    html.Br(),
    html.Hr(),
#Format the dashboard so that pie chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            ),
        html.Div(
            id='map-id',
            className='col s12 m6'
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

#Callback for updating dashboard based on radio button filter click
@app.callback([Output('datatable-id','data'),
              Output('datatable-id', 'columns')],
              [Input('filter-type', 'value')])
def update_dashboard(value):
    #initalize empty query
    query = {}
    
    #if radio button value is 'All', no query parameters
    if value == 'All':
        query ={}
        
    #if radio button value is 'Water', query suitable Water Rescue dogs
    elif value == 'Water':
        query = {
            #use re.compile to pattern match dog (catch abbreviations and mixes) breeds within the dataset
            'breed': {'$in': [re.compile('.*Lab.*'), re.compile('.*Ches.*'), re.compile('.*Newfound.*')]
                     },
            #filter only displays dogs that are available (i.e. euthanized will not be displayed)
            'outcome_type': {'$in': ['Adoption', 'Rto-Adoption', 'Transfer', 'Relocate']},
            #filters desired sex
            'sex_upon_outcome': 'Intact Female',
            #filters desired age
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
        
    #if radio button value is 'Mountain', query suitable Mountain/Wilderness Rescue dogs
    elif value == 'Mountain':
        query = {
            'breed': {'$in': [re.compile('.*German.*'), re.compile('.*Alaskan Mala.*'),
                              re.compile('.*Old English.*'), re.compile('.*Husky.*'),
                              re.compile('.*Rott.*')]
                     },
            'outcome_type': {'$in': ['Adoption', 'Rto-Adoption', 'Transfer', 'Relocate']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
        
    #if radio button value is 'Disaster', query suitable Disaster/Individual tracking dogs
    elif value == 'Disaster':
        query = {
            'breed': {'$in': [re.compile('.*Dober.*'), re.compile('.*German.*'), re.compile('.*Golden Retr.*'),
                             re.compile('.*Bloodh.*'), re.compile('.*Rottweiler.*')]
                     },
            'outcome_type': {'$in': ['Adoption', 'Rto-Adoption', 'Transfer', 'Relocate']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    #Stores the database read query in dataframe
    df = pd.DataFrame.from_records(db.read(query))
    #Drop 'id' column to prevent table crash
    df.drop(columns=['_id'], inplace=True)
    #specify columns to be returned from callback
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    #Set return data as dataframe dictionary
    data = df.to_dict("records")

    return (data, columns)
    

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')])
def update_graphs(viewData):
    # Prevent graph update if there is no data passed to callback
    if viewData is None:
        raise PreventUpdate
    
    #Load dataframe
    dff = pd.DataFrame.from_dict(viewData)
    
    #Setting value counts for breeds in the breed column
    breeds = dff['breed'].value_counts()
    #Creating pie object with breed names as key and breed counts as pie chart values
    figure = px.pie(values=breeds.values, names=breeds.index)
    #Set value texts inside of pie chart
    figure.update_traces(textposition='inside')
    #Set text size and hide text that will not fit in chart
    figure.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
    #return the graph
    return [
        dcc.Graph(
            figure=figure
        ) 
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
        return [{
            'if': { 'column_id': i },
            'background_color': '#D2F3FF'
        } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_viewport_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', 'derived_viewport_data'),
     Input('datatable-id', 'derived_viewport_selected_rows')]
)
def update_map(viewData, index):
    #If datatable is empty, prevent map from updating
    if viewData is None:
        raise PreventUpdate
    #If selected_row index is empty, prevent map from updating
    elif index is None:
        raise PreventUpdate
    
    #If length of viewport selected row indexes is zero, prevent update
    #Prevents index out of bounds when a user selects a row and then selects a filter
    if (len(index) == 0):
        raise PreventUpdate
        
    #Load datatable into dataframe
    dff = pd.DataFrame.from_dict(viewData)
    
    #Set row index
    if index is None:
        row = 0
    else:
        row = index[0]
        
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]



app.run_server(debug=True)


Dash app running on http://127.0.0.1:13102/
